# Qwen3-TTS Voice Cloning Inference (MLX)
Run on Mac Apple Silicon. Uses `mlx-audio` for native Metal acceleration.

In [ ]:
from pathlib import Path

REF_AUDIO = "processed/mic_recording_qwen3tts_ref.wav"  # output from inference_preprocess.ipynb
REF_TEXT = "When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arch, with its path high above, and its two ends apparently beyond the horizon. There is, according to legend, a boiling pot of gold at one end. People look, but no one ever finds it. When a man looks for something beyond his reach, his friends say he is looking for the pot of gold at the end of the rainbow. Throughout history, the rainbow has been a symbol of hope and a sign of things to come. The vibrant bands of red, orange, yellow, green, blue, and violet curve gracefully across the sky, reminding us of the calm that follows a storm. Scientists observe these wavelengths to understand the physics of light, while artists simply try to capture their fleeting brilliance on canvas."
MODEL_ID = "mlx-community/Qwen3-TTS-12Hz-1.7B-Base-8bit"
OUTPUT_DIR = Path("tts_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ref audio: {REF_AUDIO}")
print(f"Model: {MODEL_ID}")

Ref audio: processed/mic_recording_qwen3tts_ref.wav
Model: mlx-community/Qwen3-TTS-12Hz-1.7B-Base-8bit


In [ ]:
from mlx_audio.tts.utils import load_model

model = load_model(MODEL_ID)
print("Model loaded.")

In [ ]:
import soundfile as sf
import numpy as np

text = "Hello, this is a test of my cloned voice running entirely on my Mac."

results = list(model.generate(
    text=text,
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,

    # =======================
    # Tuneable parameters

    # temperature=0.8,        # lower = more robotic, higher = more varied. Default ~1.0
    # top_p=0.9,              # nucleus sampling threshold
    # top_k=30,               # limits token pool per step
    # repetition_penalty=1.05, # prevents loops. 1.05-1.1 recommended
    # max_tokens=4096,        # safety cap to prevent infinite generation
))

audio = np.array(results[0].audio)
sr = 24000
out_path = OUTPUT_DIR / "clone_test.wav"
sf.write(str(out_path), audio, sr)
print(f"Saved: {out_path} ({len(audio)/sr:.2f}s)")

Saved: tts_output/clone_test.wav (4.40s)


In [4]:
texts = [
    "The weather is looking great today.",
    "I'm excited to share this project with the team.",
    "Let me know if you have any questions about the results.",
]

for i, text in enumerate(texts):
    results = list(model.generate(
        text=text,
        ref_audio=REF_AUDIO,
        ref_text=REF_TEXT,
    ))
    audio = np.array(results[0].audio)
    out_path = OUTPUT_DIR / f"clone_batch_{i:03d}.wav"
    sf.write(str(out_path), audio, sr)
    print(f"[{i}] {out_path.name} ({len(audio)/sr:.2f}s) | {text[:50]}")

print(f"\nAll saved to {OUTPUT_DIR}/")

[0] clone_batch_000.wav (1.52s) | The weather is looking great today.
[1] clone_batch_001.wav (2.32s) | I'm excited to share this project with the team.
[2] clone_batch_002.wav (2.96s) | Let me know if you have any questions about the re

All saved to tts_output/


In [8]:
text = "Twas brillig, and the slithy toves. Did gyre and gimble in the wabe, All mimsy were the borogoves, NIGGER And the mome raths outgrabe.Beware the Jabberwock, my son! The jaws that bite, the claws that catch! Beware the Jubjub bird, and shun The frumious Bandersnatch!"

audio_chunks = []
for result in model.generate(
    text=text,
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    stream=True,
    streaming_interval=0.32,
):
    audio_chunks.append(np.array(result.audio))

audio = np.concatenate(audio_chunks)
out_path = OUTPUT_DIR / "clone_stream.wav"
sf.write(str(out_path), audio, sr)
print(f"Saved: {out_path} ({len(audio)/sr:.2f}s, {len(audio_chunks)} chunks)")

Saved: tts_output/clone_stream.wav (19.44s, 61 chunks)
